# Tiny RAG

Use the TinyLlama model to build a simple RAG system to answer a question about an event that was outside of the model's knowledge cutoff.


---
**Requirements**

A GPU runtime: 4-bit loading with bitsandbytes needs a CUDA GPU. In Colab, use Runtime → Change runtime type → T4 GPU.

The notebook downloads the sample PDF (`./data/2024–25_NFL_playoffs_pg_1.pdf`) from this repo if it is not already there.

---

## Model Info
model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
format: int4 (4-bit quantized)
size on disk: ~700MB
ram used: ~1GB

TinyLlama-1.1B-Chat-v1.0 is a compact, 1.1 billion–parameter language model trained on 3 trillion tokens and fine-tuned for conversational use with chat-style alignment. Built to mirror the LLaMA architecture, it delivers strong performance for tasks like RAG and assistant-style Q&A while remaining efficient enough for edge devices and Colab environments.


## Vector DB
FAISS
http://github.com/facebookresearch/faiss


FAISS (Facebook AI Similarity Search) is an open-source library for fast, efficient vector similarity search and clustering of dense embeddings. It’s commonly used in RAG systems to retrieve relevant documents by comparing query vectors against millions of stored embeddings using various indexing strategies.


## Embeddings
all-MiniLM-L6-v2 sentence transformer

all-MiniLM-L6-v2 is a lightweight sentence transformer model that maps text to 384-dimensional embeddings optimized for semantic similarity tasks like clustering, retrieval, and ranking. Trained using a distillation approach from larger transformer models, it offers an excellent balance of speed and accuracy for real-time applications.

## Document Processing
PyMuPDF for PDF text extraction

PyMuPDF is a Python library that provides fast, lightweight access to PDF and other document formats, enabling text extraction, page manipulation, and metadata handling. Built on the MuPDF engine, it supports precise layout-aware text extraction, making it ideal for preprocessing PDFs in NLP and document analysis workflows.

## Install requirements

In [1]:
!pip install transformers==5.23.0 accelerate==1.16.0 bitsandbytes==0.50.2 sentence-transformers==6.3.0 faiss-cpu==1.16.0 PyMuPDF==2.1

## Imports

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import faiss
import pymupdf as fitz
from sentence_transformers import SentenceTransformer

## Model

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")

## Test Pre-RAG
Let's see what the model knows about our topic before we give access to new information via RAG.

We will ask the model which team won the 2024 NFL Championship.

Correct answer: The Philadelphia Eagles won the 2024 NFL Championship on February 9, 2025

But the model we're using has a knowledge cut-off for this model is somewhere in late 2023 so there is no chance it was trained on this information.

In [4]:
query = "Who won the 2024 NFL Championship?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Question: {query}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])


The 2024 NFL Championship will be the Super Bowl, which will be held in the United States.


### Result? It doesn't know
The model doesn't name a winner. It talks about the 2024 championship as if it hasn't happened yet, because its training data ends before the game was played.

The answer is the Philadelphia Eagles: https://en.wikipedia.org/wiki/2024%E2%80%9325_NFL_playoffs

Let's add some "Context" for the model to fix this issue!

In [5]:
context = """The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history."""

### NOTE
As you can see the "context" is just a string with some information from wikipedia about the 2024 Super Bowl Winner.

We cut a few corners here...

Normally there is a vector database that stores information that is used for context. That knowledge comes from text, PDFs, and other documents and usually has to be processed, chunked, and inserted into a Vector DB.

Instead we're using just this string and we're going to add it to our prompt as context to show the effect of setting up an actual Vector DB to our query.

In [6]:
query = "Who won the 2024 NFL Championship?"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Context: {context} Question: {query}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])


The 2024 NFL Championship was won by the Philadelphia Eagles, who defeated the defending champion Kansas City Chiefs 40-22 in the Super Bowl LIX.


As you can see the model now returns the correct information thanks to the added context.

Let's add an actual Vector DB and setup a proper RAG system.

## RAG Setup
We're going to use FAISS as our Vector DB and we're going to add the contents of a first page of the PDF print out of the wikipedia article about the 2024-2025 NFL playoffs which contains the information we need.

### Get the PDF
Download the sample PDF from the GitHub repo if it isn't already in `./data`.

In [7]:
import os, urllib.request

pdf_path = "./data/2024–25_NFL_playoffs_pg_1.pdf"
if not os.path.exists(pdf_path):
    os.makedirs("./data", exist_ok=True)
    url = "https://raw.githubusercontent.com/didiergarcia/tiny-rag/main/data/2024%E2%80%9325_NFL_playoffs_pg_1.pdf"
    urllib.request.urlretrieve(url, pdf_path)

### Extract text from the PDF
We need to get at the text in the PDF. We'll use "fitz" to extract the text.

In [8]:
pdf_text = "\n".join(block[4] for block in fitz.open("./data/2024–25_NFL_playoffs_pg_1.pdf").load_page(0).get_text("blocks"))

Let's look at the extracted text.

In [9]:
pdf_text

"2024–25 NFL playoffs\n\nDates\nJanuary 11 – February 9,\n2025\n\nSeason\n2024\n\nTeams\n14\n\nGames played\n13\n\nSuper Bowl LIX\nsite\n\nCaesars Superdome\n\nNew Orleans, Louisiana\n\nDefending\nchampions\n\nKansas City Chiefs\n\nChampion\nPhiladelphia Eagles\n\nRunner-up\nKansas City Chiefs\n\nConference\nrunners-up\n\nBuffalo Bills\n\nWashington\nCommanders\n\nNFL playoffs\n← 2023–24\n\n2024–25 NFL playoffs\n\nThe NFL playoffs for the 2024 season began on January 11, 2025, and\nconcluded with Super Bowl LIX on February 9 at Caesars Superdome in\nNew Orleans, Louisiana, when the Philadelphia Eagles defeated the\ndefending champion Kansas City Chiefs 40–22 to win their second Super\nBowl in franchise history.\n\nAll playoff teams in both conferences won at least 10 games, the first time\nsince 2012 and the first time since the NFL playoffs expanded to a 14-\nteam format in 2020. This was the second time since their respective\nmoves to Los Angeles where the Rams and Chargers both qua

### Chunk the Text
We need to break up the text into smaller chunks to help find specific bits of information we're looking for and to fit within the embeddings length for the Vector DB.

In [10]:
chunk_size = 150
pdf_word_chunks = pdf_text.split() # Split on whitespace
pdf_chunks = [" ".join(pdf_word_chunks[i:i+chunk_size]) for i in range(0, len(pdf_word_chunks), chunk_size)]

print(f"{len(pdf_chunks)} chunks")

4 chunks


Let's look at the chunks.

In [11]:
pdf_chunks

['2024–25 NFL playoffs Dates January 11 – February 9, 2025 Season 2024 Teams 14 Games played 13 Super Bowl LIX site Caesars Superdome New Orleans, Louisiana Defending champions Kansas City Chiefs Champion Philadelphia Eagles Runner-up Kansas City Chiefs Conference runners-up Buffalo Bills Washington Commanders NFL playoffs ← 2023–24 2024–25 NFL playoffs The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history. All playoff teams in both conferences won at least 10 games, the first time since 2012 and the first time since the NFL playoffs expanded to a 14- team format in 2020. This was the second time since their respective moves to Los Angeles where the Rams and Chargers both',
 'qualified for the playoffs, after 2018. Within each conference, the fou

### Create Embeddings
We create Embeddings for each chunk. Embeddings are used to estimate the meaning of the chunk. When two embeddings are close to each other in geometrically in multi-dimensional space we know the chunks are close in meaning.

In [12]:
sentenceTransformerModel = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentenceTransformerModel.encode(pdf_chunks, convert_to_tensor=False)

### Insert into Vector DB
Let's add our embeddings to the Vector DB so we can later query it.

In [13]:
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

### Test the Vector DB
Let's see if asking about the Championship brings up the right chunk of text.

In [14]:
query = "Who won the 2024 NFL Championship?"
q_embedding = sentenceTransformerModel.encode([query])
D, I = index.search(q_embedding, k=3)

for idx in I[0]:
    print("Match:", pdf_chunks[idx])

Match: 2024–25 NFL playoffs Dates January 11 – February 9, 2025 Season 2024 Teams 14 Games played 13 Super Bowl LIX site Caesars Superdome New Orleans, Louisiana Defending champions Kansas City Chiefs Champion Philadelphia Eagles Runner-up Kansas City Chiefs Conference runners-up Buffalo Bills Washington Commanders NFL playoffs ← 2023–24 2024–25 NFL playoffs The NFL playoffs for the 2024 season began on January 11, 2025, and concluded with Super Bowl LIX on February 9 at Caesars Superdome in New Orleans, Louisiana, when the Philadelphia Eagles defeated the defending champion Kansas City Chiefs 40–22 to win their second Super Bowl in franchise history. All playoff teams in both conferences won at least 10 games, the first time since 2012 and the first time since the NFL playoffs expanded to a 14- team format in 2020. This was the second time since their respective moves to Los Angeles where the Rams and Chargers both
Match: other two surviving teams play each other, with the higher-seed

As expected the first match is the chunk containing the information we need for the context!

## Query with RAG Context
We've now got a Vector DB setup to do our Retrieval and we can try our initial query again.

In [15]:
query = "Who won the 2024 NFL Championship?"
q_embedding = sentenceTransformerModel.encode([query])
D, I = index.search(q_embedding, k=3)
context = pdf_chunks[I[0][0]]

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": f"Context: {context} Question: {query}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300)
decoded_output = tokenizer.decode(out[0], skip_special_tokens=True)
print(decoded_output.split("<|assistant|>")[1])


The 2024 NFL Championship was won by the Philadelphia Eagles, who defeated the defending champion Kansas City Chiefs 40-22 in Super Bowl LIX at Caesars Superdome in New Orleans, Louisiana.


**Success!** The model is now aware of data from outside of its original training data and can now successfully answer the question!


## Memory used
How much GPU memory did this take? `get_memory_footprint()` is the size of the loaded 4-bit model; the peak is the most memory PyTorch allocated at any point in this notebook, including generation.

In [16]:
print(f"Model weights in memory: {model.get_memory_footprint() / 1024**3:.2f} GB")
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

Model weights in memory: 0.70 GB
Peak GPU memory allocated: 0.89 GB


## The same pipeline with the `tiny_rag` package
Everything above is also packaged as a small library in this repo (`src/tiny_rag`). Install it, then build the index from the same PDF and ask the same question. This loads a second copy of the model, so it is not part of the memory figure above.

In [17]:
!pip install -q "tiny-rag[quant,faiss] @ git+https://github.com/didiergarcia/tiny-rag"

`Index.from_files` extracts the PDF text (with pypdf), chunks it and embeds it. `RAG.ask` retrieves the top `k` chunks, puts them into the prompt with the model's chat template, and returns the answer together with the chunks it used.

In [18]:
from tiny_rag import RAG, HFTransformersLLM, Index

rag_index = Index.from_files([pdf_path], chunk_size=150)
rag = RAG(rag_index, HFTransformersLLM(load_in_4bit=True))

answer = rag.ask("Who won the 2024 NFL Championship?", k=2)
print(answer.text)
for hit in answer.sources:
    print(f"{hit.score:.3f}  {hit.chunk.source}  {hit.chunk.text[:60]}...")

The 2024 NFL Championship was won by the Philadelphia Eagles, who defeated the defending champion Kansas City Chiefs 40-22 in the Super Bowl LIX at Caesars Superdome in New Orleans, Louisiana.
0.596  data/2024–25_NFL_playoffs_pg_1.pdf#page=1  2024–25 NFL playoffs Dates January 11 – February 9, 2025 Sea...
0.499  data/2024–25_NFL_playoffs_pg_1.pdf#page=1  other two surviving teams play each other, with the higher-s...
